In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# 1. CONFIGURACIÓN DE RUTAS Y CARGA
BASE = Path("../data")
df = pd.read_csv(BASE / "processed" / "energia_limpia.csv")
df['datetime_clean'] = pd.to_datetime(df['datetime_clean'])

# 2. INGENIERÍA DE CARACTERÍSTICAS
df['hora'] = df['datetime_clean'].dt.hour
df['mes'] = df['datetime_clean'].dt.month
df['dia_semana'] = df['datetime_clean'].dt.dayofweek

# Definimos variables predictoras y objetivo
columnas_predictoras = [
    'hora', 'mes', 'dia_semana',
    'pvgis_meteo_var_0', 'pvgis_meteo_var_1', 
    'tmed', 'sol'
]
columna_objetivo = 'solar_esios_kwh'

# --- SOLUCIÓN DE MUESTRAS VACÍAS (IMPUTACIÓN INTELIGENTE) ---
# Primero, eliminamos filas únicamente si falta el objetivo a predecir (generación solar)
df_ml = df.dropna(subset=[columna_objetivo]).copy()

# Forzamos conversión numérica por seguridad
for col in columnas_predictoras:
    if col in df_ml.columns:
        df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')

# Rellenamos los vacíos de clima usando interpolación lineal (ideal para series temporales)
df_ml[columnas_predictoras] = df_ml[columnas_predictoras].interpolate(method='linear', limit_direction='both')

# Si queda algún nulo residual aislado, lo rellenamos con la media de esa columna
for col in columnas_predictoras:
    if df_ml[col].isnull().sum() > 0:
        df_ml[col] = df_ml[col].fillna(df_ml[col].mean())

# Definimos nuestras matrices de entrenamiento finales
X = df_ml[columnas_predictoras]
y = df_ml[columna_objetivo]

print(f"📊 Dataset listo de forma segura para Machine Learning.")
print(f"👉 Cantidad de muestras recuperadas para entrenar: {X.shape[0]} filas.")

# 3. DIVISIÓN Y ENTRENAMIENTO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n🚀 Entrenando modelo predictor fotovoltaico (Random Forest)...")
modelo_solar = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_solar.fit(X_train, y_train)
print("¡Entrenamiento completado!")

# 4. EVALUACIÓN DE PRECISIÓN
predicciones = modelo_solar.predict(X_test)
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"\n📊 MÉTRICAS DE PRECISIÓN DE START ENERGY AI:")
print(f"👉 Error Absoluto Medio (MAE): {mae:.2f} kWh")
print(f"👉 Coeficiente R² (Score de precisión): {r2:.4f}")

# 5. GUARDADO DEL MODELO PARA EL BACKEND
ruta_modelos = Path("../backend/models")
ruta_modelos.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_solar, ruta_modelos / "predictor_solar.joblib")
print(f"\n💾 ¡Modelo exportado correctamente en: {ruta_modelos / 'predictor_solar.joblib'}!")


📊 Dataset listo de forma segura para Machine Learning.
👉 Cantidad de muestras recuperadas para entrenar: 8931 filas.

🚀 Entrenando modelo predictor fotovoltaico (Random Forest)...
¡Entrenamiento completado!

📊 MÉTRICAS DE PRECISIÓN DE START ENERGY AI:
👉 Error Absoluto Medio (MAE): 299.94 kWh
👉 Coeficiente R² (Score de precisión): 0.1368

💾 ¡Modelo exportado correctamente en: ..\backend\models\predictor_solar.joblib!
